In [3]:
# !pip install --quiet torch torchvision torchaudio

# !pip install --quiet opencv-python matplotlib Pillow

# !pip install --quiet git+https://github.com/facebookresearch/segment-anything.git



In [4]:
# !pip install git+https://github.com/facebookresearch/segment-anything.git
# !pip install opencv-python-headless
# !pip install torch torchvision

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

image_path = '/kaggle/input/badqualityphoto/WhatsApp Image 2024-12-07 at 07.58.08_12daec15.jpg'  
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

input_size = (1280, 1280)
resized_image = cv2.resize(image, input_size)

# Display the original and resized images
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(image)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Resized Image")
plt.imshow(resized_image)
plt.axis('off')

plt.show()





NameError: name 'image' is not defined

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
sam_checkpoint = "/kaggle/input/hcheckpoint/sam_vit_h_4b8939.pth" 
model_type = "vit_h"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)

predictor.set_image(resized_image)

input_point = np.array([[500, 500]]) 
input_label = np.array([1]) 



cuda


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/hcheckpoint/sam_vit_h_4b8939.pth'

In [ ]:
masks, scores, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

plt.figure(figsize=(10, 10))
for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.subplot(1, len(masks), i + 1)
    plt.imshow(resized_image)
    plt.imshow(mask, alpha=1) 
    plt.title(f"Mask {i+1} - Score: {score:.3f}")
    plt.axis('off')

plt.show()

print(masks)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_with_boxes = resized_image.copy()

for i, mask in enumerate(masks):
    mask_uint8 = (mask * 255).astype(np.uint8)

    kernel = np.ones((5, 5), np.uint8)
    refined_mask = cv2.morphologyEx(mask_uint8, cv2.MORPH_CLOSE, kernel)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_OPEN, kernel)

    blurred_mask = cv2.GaussianBlur(refined_mask, (5, 5), 0)

    contours, _ = cv2.findContours(blurred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    min_area = 1000  
    for contour in contours:
        if cv2.contourArea(contour) > min_area:
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(image_with_boxes, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.figure(figsize=(10, 10))
plt.title("Bounding Boxes for Each Refined Mask")
plt.imshow(image_with_boxes)
plt.axis('off')
plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_with_boxes = resized_image.copy()

for i, mask in enumerate(masks):
    mask_uint8 = (mask * 255).astype(np.uint8)

    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(image_with_boxes, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.figure(figsize=(10, 10))
plt.title("Bounding Boxes for Each Mask")
plt.imshow(image_with_boxes)
plt.axis('off')
plt.show()

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

save_dir = "/kaggle/working/cropped_images/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

cropped_images = []

for idx, contour in enumerate(contours):
    min_area = 1000
    if cv2.contourArea(contour) > min_area:
        x, y, w, h = cv2.boundingRect(contour)
        
        cropped_image = resized_image[y:y + h, x:x + w]
        cropped_images.append(cropped_image)
        
        save_path = os.path.join(save_dir, f"cropped_image_{idx + 1}.png")
        cv2.imwrite(save_path, cv2.cvtColor(cropped_image, cv2.COLOR_RGB2BGR))

plt.figure(figsize=(15, 10))

for i, cropped_image in enumerate(cropped_images):
    plt.subplot(1, len(cropped_images), i + 1)
    plt.imshow(cropped_image)
    plt.title(f"Cropped Image {i + 1}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_with_boxes = resized_image.copy()

for i, mask i
    mask_uint8 = (mask * 255).astype(np.uint8)
    inverted_mask = cv2.bitwise_not(mask_uint8)

    kernel = np.ones((5, 5), np.uint8)
    refined_mask = cv2.morphologyEx(inverted_mask, cv2.MORPH_CLOSE, kernel)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_OPEN, kernel)

    blurred_mask = cv2.GaussianBlur(refined_mask, (5, 5), 0)

    contours, _ = cv2.findContours(blurred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    min_area = 1000
    for contour in contours:
        if cv2.contourArea(contour) > min_area:
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(image_with_boxes, (x, y), (x + w, y + h), (255, 0, 0), 2)  
            
# Display the image with bounding boxes
plt.figure(figsize=(10, 10))
plt.title("Bounding Boxes for Inverted Mask Logic")
plt.imshow(image_with_boxes)
plt.axis('off')
plt.show()


In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

save_dir = "/kaggle/working/cropped_images_3/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

cropped_images = []

for idx, contour in enumerate(contours):

    min_area = 1000
    if cv2.contourArea(contour) > min_area:
        
        x, y, w, h = cv2.boundingRect(contour)
        

        cropped_image = resized_image[y:y + h, x:x + w]
        cropped_images.append(cropped_image)
        
    
        save_path = os.path.join(save_dir, f"cropped_image_{idx + 1}.png")
        cv2.imwrite(save_path, cv2.cvtColor(cropped_image, cv2.COLOR_RGB2BGR)) 

plt.figure(figsize=(15, 10))

for i, cropped_image in enumerate(cropped_images):
    plt.subplot(1, len(cropped_images), i + 1)
    plt.imshow(cropped_image)
    plt.title(f"Cropped Image {i + 1}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np
import matplotlib.pyplot as plt

model = load_model('/kaggle/input/efficientnetb0-classi/efficientnetb0_classi.h5')
print("Model loaded successfully!")


In [ ]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np

def preprocess_image(image_path, target_size=(190, 190)):
    image = cv2.imread(image_path)
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    image_resized = cv2.resize(image, target_size)
    
    image_normalized = image_resized / 255.0
    
    input_tensor = np.expand_dims(image_normalized, axis=0)
    return input_tensor

def predict_image_class(model, image_path, class_labels):
    input_tensor = preprocess_image(image_path)

    predictions = model.predict(input_tensor)
    
    predicted_index = np.argmax(predictions, axis=1)[0]
    confidence = predictions[0][predicted_index]
    predicted_label = class_labels[predicted_index]
    return predicted_label, confidence


model_path = '/kaggle/input/efficientnetb0-classi/efficientnetb0_classi.h5'  
model = load_model(model_path)
print("Model loaded successfully!")

class_labels = ['Asian-Green-Bee-Eater', 'Brown-Headed-Barbet', 'Cattle-Egret', 
                'Common-Kingfisher', 'Common-Myna', 'Common-Rosefinch', 'Common-Tailorbird', 
                'Coppersmith-Barbet', 'Forest-Wagtail', 'Gray-Wagtail', 'Hoopoe', 'House-Crow', 
                'Indian-Grey-Hornbill', 'Indian-Peacock', 'Indian-Pitta', 'Indian-Roller', 
                'Jungle-Babbler', 'Northern-Lapwing', 'Red-Wattled-Lapwing', 'Ruddy-Shelduck', 
                'Rufous-Treepie', 'Sarus-Crane', 'White-Breasted-Kingfisher', 'White-Breasted-Waterhen', 
                'White-Wagtail']

image_path = '/kaggle/working/cropped_images_2/cropped_image_2.png'  

predicted_label, confidence = predict_image_class(model, image_path, class_labels)


print(f"Predicted Class: {predicted_label}")
print(f"Confidence: {confidence:.2f}")


**Our Model Based on EfficientNetB0 Architecture and Model Trained from Scratch**


In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

def preprocess_image(image_path, target_size=(190, 190)):
    

    image = cv2.imread(image_path)
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    image_resized = cv2.resize(image, target_size)
   
    image_normalized = image_resized / 255.0
    
    input_tensor = np.expand_dims(image_normalized, axis=0)
    return input_tensor, image_resized  

def predict_image_class(model, image_path, class_labels):
    
    input_tensor, resized_image = preprocess_image(image_path)
    
    predictions = model.predict(input_tensor)
    
    predicted_index = np.argmax(predictions, axis=1)[0]
    confidence = predictions[0][predicted_index]
    predicted_label = class_labels[predicted_index]
    return predicted_label, confidence, resized_image


model_path = '/kaggle/input/efficientnetb0-classi/efficientnetb0_classi.h5'  
model = load_model(model_path)
print("Model loaded successfully!")

class_labels = ['Asian-Green-Bee-Eater', 'Brown-Headed-Barbet', 'Cattle-Egret', 
                'Common-Kingfisher', 'Common-Myna', 'Common-Rosefinch', 'Common-Tailorbird', 
                'Coppersmith-Barbet', 'Forest-Wagtail', 'Gray-Wagtail', 'Hoopoe', 'House-Crow', 
                'Indian-Grey-Hornbill', 'Indian-Peacock', 'Indian-Pitta', 'Indian-Roller', 
                'Jungle-Babbler', 'Northern-Lapwing', 'Red-Wattled-Lapwing', 'Ruddy-Shelduck', 
                'Rufous-Treepie', 'Sarus-Crane', 'White-Breasted-Kingfisher', 'White-Breasted-Waterhen', 
                'White-Wagtail']

image_folder = '/kaggle/working/cropped_images_3/'
image_paths = [os.path.join(image_folder, fname) for fname in os.listdir(image_folder) if fname.endswith('.png')]

MIN_CONFIDENCE = 0.6 


plt.figure(figsize=(15, len(image_paths) * 3)) 
valid_images = 0  

for i, image_path in enumerate(image_paths):
    
    predicted_label, confidence, resized_image = predict_image_class(model, image_path, class_labels)
    
    
    if confidence < MIN_CONFIDENCE:
        print(f"Skipping image {image_path} due to low confidence ({confidence:.2f})")
        continue
    
    
    plt.subplot(len(image_paths), 1, valid_images + 1)
    plt.imshow(resized_image)
    plt.title(f"Predicted: {predicted_label} (Confidence: {confidence:.2f})")
    plt.axis('off')
    valid_images += 1

if valid_images > 0:
    plt.tight_layout()
    plt.show()
else:
    print("No images met the minimum confidence threshold.")


In [ ]:
import tensorflow as tf
import cv2
import numpy as np

def preprocess_image(image_path, target_size=(190, 190)):
    
    image = cv2.imread(image_path)
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    image_resized = cv2.resize(image, target_size)
    
    image_normalized = image_resized / 255.0
    
    input_tensor = np.expand_dims(image_normalized, axis=0)
    return input_tensor, image_resized

def predict_image_class(model, image_path, class_labels):
    
    input_tensor, resized_image = preprocess_image(image_path)
    
    predictions = model.predict(input_tensor)
    predicted_index = np.argmax(predictions, axis=1)[0]
    confidence = predictions[0][predicted_index]
    predicted_label = class_labels[predicted_index]
    return predicted_label, confidence, resized_image


model_path = 'fine_tuned_pretrained_efficientnetb0.h5'
model = tf.keras.models.load_model(model_path)
print("Model loaded successfully!")

class_labels = [
    'Asian-Green-Bee-Eater', 'Brown-Headed-Barbet', 'Cattle-Egret', 'Common-Kingfisher', 'Common-Myna', 
    'Common-Rosefinch', 'Common-Tailorbird', 'Coppersmith-Barbet', 'Forest-Wagtail', 'Gray-Wagtail', 
    'Hoopoe', 'House-Crow', 'Indian-Grey-Hornbill', 'Indian-Peacock', 'Indian-Pitta', 'Indian-Roller', 
    'Jungle-Babbler', 'Northern-Lapwing', 'Red-Wattled-Lapwing', 'Ruddy-Shelduck', 'Rufous-Treepie', 
    'Sarus-Crane', 'White-Breasted-Kingfisher', 'White-Breasted-Waterhen', 'White-Wagtail'
]

image_path = '/kaggle/input/indian-birds/Birds_25/train/Hoopoe/Hoopoe_1.jpg' 


predicted_label, confidence, resized_image = predict_image_class(model, image_path, class_labels)


print(f"Predicted Class: {predicted_label}")
print(f"Confidence: {confidence:.2f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.imshow(resized_image)
plt.title(f"Predicted: {predicted_label} (Confidence: {confidence:.2f})")
plt.axis('off')
plt.show()
